### Geometry Model

The model is 2D box 10000 x 2900 km in dimensions, and chosen to be large enough to avoid boundary effects.

```
subsection Geometry model
  set Model name = box
    subsection Box
      set X extent = 10000e3
      set Y extent = 2900e3
   end
end
```

## Modeling Subduction

To model subduction zones, we need to model a slab that is denser than the surrounding mantle. Additionally, we may need a weak layer above the slab (representative of ocean sediments) that  decouples it from the overriding plate.

The model setup below is based on the paper by [Garel et al. (2014)](https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1002%2F2014GC005257) (figure 1) and  consists of two plates: a 100 Ma old subducting plate beneath an overriding 40 Ma old plate.

We use [Geodynamic World Builder](https://geodynamicworldbuilder.github.io/) (GWB) to define the initial temperature and compositional fields representing the two plates and the weak-zone above it, and then use ASPECT to solve the evolution of the system.

### Initial Temperature

The initial temperature field is defined using a half-space cooling model for the subducting and overriding plates, implemented via GWB (details in the [appendix](#appendix)).

```
subsection Initial temperature model
  set List of model names =   world builder
end
```

### Initial Compositional Field

We use a 10 km thick weak layer above the subducting plate, representative of asthe ocean sediments, to allow the decoupled slab to subduct using a compositional field named `weak_zone`. Additionally, we also represent lower mantle using a `static` compositional field to account for its different mechanical properties, as defined in Garel et al., (2014) Table 1. More details on the implementation using GWB can be found in the [appendix](#appendix).

```
subsection Compositional fields
   set Number of fields = 2
   set Names of fields  = lower_mantle, weak_zone
   set Types of fields  =  chemical composition, chemical composition
   set Compositional field methods =  static, field
end
```

<div align="left">
<img src='./images/model_setup.png' width="800"/>
<figcaption align = "left"> Location of the two compositional fields described in our model setup with a cut showing the model discretization. </figcaption>
</div>


The geometry of the `weak_zone` is described using GWB, while we use a depth-based function to define the `lower_mantle` compositional field.

```
subsection Initial composition model
  set List of model names =  function,  world builder
  set List of model operators =  maximum
  
  subsection Function
    set Variable names      = depth, nan
    set Coordinate system   = depth
    set Function expression = if (depth > 660e3, 1, 0); 0
  end
end
```

### Boundary Velocity Model

We apply a constant velocity of 3 cm/yr at the top boundary of the subducting plate, and keep the other boundaries free-slip. This is done to simulate plate generation from a ridge, assumed to beat the left side of the model.  

```
subsection Boundary velocity model
  set Prescribed velocity boundary indicators = top:function
  set Tangential velocity boundary indicators = left, right, bottom

  subsection Function
    # We choose a half-spreading rate of u0=3cm/yr.
    set Function constants  = u0=0.03, x1= 5000e3
    set Variable names      = x,y
    set Function expression = if (x < x1, u0, 0); 0
  end
end
```

> Note: The study by Garel et al. (2014) uses a free-surface at the top boundary. However, in this tutorial we use an imposed velocity boundary condition to avoid the complexities of modeling a free-surface, and to focus on the subduction process itself.

### Material Model

The material model derives a composite viscosity based on the four different mechanisms: diffusion creep (low stress), dislocation creep (high stress), plastic yielding limiting stress at low temperatures, and Peierls creep at low-temperature and high-stress conditions. The parameters for the different mechanisms are based on the values given in Table 1 of Garel et al. (2014), and are implemented in ASPECT using the `visco plastic` material model.

A subset of the material parameters are shown below, and the full list of parameters can be found in the accompanying parameter file.

```
subsection Material model
  set Model name =    visco plastic

  subsection Visco Plastic
    set Activation energies for Peierls creep = 540e3
    set Activation volumes for Peierls creep  = 10e-6
    set Prefactors for Peierls creep          = 1e-150, 1e-300, 1e-150
    set Stress exponents for Peierls creep    = 20

    set Activation energies for diffusion creep = 300e3, 200e3, 300e3
    set Activation volumes for diffusion creep  = 4e-6,  1.5e-6, 4e-6
    set Prefactors for diffusion creep          = 3e-11, 6e-17, 3e-11
    set Stress exponents for diffusion creep    = 1
    set Grain size exponents for diffusion creep = 0
  end
end   
```

<a name="appendix"></a>
## Appendix: Geodynamic World Builder Setup

We use GWB to define the initial temperature and compositional fields in our model setup, as well as the weak zone above the slab, using the following components:

- **Overriding Plate** (`oceanic plate` feature, up to 100 km depth)
Occupies the eastern half of the domain (x = 5,000–10,000 km). Its thermal structure is computed using a half-space cooling model, with spreading velocity of 12.5 cm/yr and a ridge located at the far-right edge (x = 10,000 km). Surface temperature is 273 K (0°C). The chosen spreading velocity corresponds to a plate age of 40 Ma at the trench.

```
    "model":"oceanic plate", "name":"Overriding Plate", "max depth":100e3,
    "coordinates":[[5000e3, 0], [10000e3, 0], [10e3, 10000e3], [5000e3, 10e3]],
    "temperature models":
        [
        {"model":"half space model",  "max depth":200e3, "spreading velocity":0.125,  
        "top temperature": 273,
        "ridge coordinates":[[[10000e3, 0], [10000e3, 10e3]]]}
        ]
```        
composition models": [{"model":"uniform", "compositions":[1], "max depth": 15e3}]
```

- **Subducting Plate** (`oceanic plate` model, up to 200 km depth)
Occupies the western half (x = 0–5,000 km). Slower spreading (5 cm/yr), with its ridge at the far-left edge (x = 0). Temperatures are defined using a half-space cooling model. The chosen spreading velocity corresponds to a plate age of 100 Ma at the trench. Surface temperature is 273 K (0°C).

```
    "model":"oceanic plate", "name":"Subducting Plate", "max depth":200e3,
    "coordinates":[[0, 0], [5000e3, 0], [5000e3, 10e3], [0, 10e3]],
    "temperature models":
        [
        {"model":"half space model",  "max depth":200e3, "spreading velocity":0.05,
        "top temperature": 273,
        "ridge coordinates":[[[0, -1e3], [0, 10e3]]]}
        ],
    "

- **The Slab** (`subducting plate` model)
The slab dips at 77° descending over a segment 336 km long and 120 km thick at the trench (x = 5,000 km). The temperature field inside the slab is defined using the `mass conserving` model that accounts for heating of the slab and cooling down of the surrounding mantle during subduction.
```
    "model":"subducting plate", "name":"Slab", "dip point": [6000e3, 0], "min depth": -1e3,
    "coordinates":[[5000e3, -10e3], [5000e3, 10e3]],
    "segments": [{"length": 336e3, "thickness": [120e3], "angle":[0, 77], "top truncation":[-100e3] }],
    "temperature models": [{"model":"mass conserving", "spreading velocity":0.05, "subducting velocity" : 0.05,
                            "ridge coordinates": [[[0, -1e3], [0, 10e3]]], "coupling depth":336e3,
                            "min distance slab top":-10e3  }],
    "composition models": [{"model":"uniform", "compositions":[1], "max distance slab top": 15e3, "min distance slab top": -1e3}]
```

- **Weak Zone** (compositional field `1`)
A 15 km thick weak layer from top of the slab and the subducting plate, representing the ocean sediments, is assigned a constant compositional field value of 1. This layer is later used in ASPECT to define a weak layer that allows the slab to decouple from the overriding plate and subduct.

### References

- Garel, F., Goes, S., Davies, D. R., Davies, J. H., Kramer, S. C., & Wilson, C. R. (2014). Interaction of subducted slabs with the mantle transition‐zone: A regime diagram from 2‐D thermo‐mechanical models with a mobile trench and an overriding plate. Geochemistry, Geophysics, Geosystems, 15(5), 1739-1765.

#### Further reading

To understand the philosophy of GWB and how to create basic tectonic features using GWB, follow the tutorial [here](https://gwb.readthedocs.io/en/latest/user_manual/basic_starter_tutorial/index.html)

 &nbsp;<div style="text-align: right">  
    &rarr; <b>NEXT: [Plot model simulation results](5_plotting_simulated_fields.ipynb) </b> <a href=""></a> &nbsp;&nbsp;
     <img src="../assets/education-gem-notebooks_icon.png" alt="icon"  style="width:4%">
  </div>